#### 문서 로드

In [1]:
from langchain_community.document_loaders import PyPDFLoader

# 문서 로드
loader = PyPDFLoader('../data/KCI_FI003153549.pdf')
documents = loader.load()

#### 문서 분할

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

splitted_documents = text_splitter.split_documents(documents)

In [4]:
from langchain_community.vectorstores import FAISS
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

#### 임베딩 모델

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 임베딩 모델 준비
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=gemini_api_key,
)

#### 임베딩 & FAISS(Facebook AI Similarity Search) 벡터스토어 생성 및 저장

##### Case1. In-memory

In [ ]:
vectorstore = FAISS.from_documents(splitted_documents, embedding_model) # 인메모리(in-memory) 상태

##### Case2. 로컬 디스크 저장(기존 파일 삭제 후 저장)

In [7]:
vectorstore = FAISS.from_documents(splitted_documents, embedding_model)

# 영구적인 파일(persistent file)**로 디스크에 저장
# 기존 폴더에 새로운 인덱스 파일을 덮어쓰기 때문에 중복된 파일이 생성되지 않음(항상 가장 마지막에 저장된 벡터스토어의 파일만 존재)
vectorstore.save_local("./faiss_index")

In [8]:
vectorstore

In [9]:
vectorstore = None

In [10]:
vectorstore

In [11]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    embedding_model,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용
)

In [12]:
vectorstore

##### Case3. 로컬 디스크 저장(기존 파일이 있을 경우 로드)

In [17]:
vectorstore = None

In [18]:
vectorstore

In [19]:
FAISS_INDEX_PATH = "./faiss_index"

if os.path.exists(FAISS_INDEX_PATH):
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embedding_model,
        allow_dangerous_deserialization=True,
    )
else:
    # FAISS 벡터스토어 생성 및 저장
    vectorstore = FAISS.from_documents(splitted_documents, embedding_model)
    vectorstore.save_local(FAISS_INDEX_PATH)

In [20]:
vectorstore

#### 검색기(Retriever)

##### Faiss 검색 메서드 작동 방식

1. 쿼리 임베딩: 사용자가 입력한 query 텍스트를 임베딩 모델을 사용하여 벡터로 변환

2. 유사성 검색: 변환된 쿼리 벡터와 벡터스토어 내의 모든 문서 벡터 간의 **거리**(distance) 또는 **유사도**(similarity)를 계산(FAISS는 이 과정에서 유클리드 거리나 코사인 유사도와 같은 알고리즘을 사용해 효율적으로 가장 가까운 벡터들을 찾음)

3. 결과 반환: 계산된 유사도 점수를 기준으로 상위 k개(이 코드에서는 k=5)의 문서 덩어리(doc)를 정렬하여 반환

##### 검색 메서드 비교

| 특징 | `as_retriever` | `similarity_search` |
| :--- | :--- | :--- |
| **반환 객체** | 검색 기능을 가진 **`Retriever` 객체** | 문서 리스트 |
| **역할** | **검색기**(객체)를 **준비** | **검색 작업**을 **실행** |
| **활용** | 복잡한 **체인**에 통합, 기능 모듈화 | **즉각적인 결과** 확인, 단독 사용 |

✨ `as_retriever`가 검색 기능을 추상화하여 제공하는 '도구'를 만드는 것이라면, `similarity_search`는 그 도구를 직접 사용하는 '행위'에 해당한다.

In [24]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

##### `similarity_search` 메서드

In [25]:

# results = vectorstore.similarity_search(query, k=3) # k는 유사도 검색에서 반환할 상위 문서 개수(top‑k)
results = vectorstore.similarity_search(query, k=5)

for idx, doc in enumerate(results, start=1):
    print(f"[결과 {idx}]\n" + doc.page_content[:300])
    print("---")

[결과 1]
의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하
기 위해 의료기기 임상시험 전문가로부터 총 158개의 문
서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음
과 같이 분류된다:
 규제 문서 (30%): FDA, EMA, PMDA 가이드라인, 
GCP 문서 등
 교육 자료 (20%): 임상시험 수행자 교육 매뉴얼, 온라
인 강의 자료 등
 프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR 
(Clinical Study Report) 템플릿 등
 의료기기 특화 문서 (15%): 의료기기 임상시험 계획
서,
---
[결과 2]
174   Journal of The Korea Society of Computer and Information 
문서(10%)로 구성되며, 각 분류는 도메인 전문가의 검토를 
통해 정확성과 신뢰성을 확보하였다. 특히, 주요 규제 기
관(FDA, EMA, PMDA) 가이드라인, GCP 문서, 환자 동의
서 템플릿 등은 모델이 국제 표준에 기반하여 학습할 수 
있도록 하였고, 교육 자료와 프로토콜은 실질적인 임상시
험 수행과 데이터 관리 작업에서 발생할 수 있는 질문들에 
대응할 수 있는 기초를 제공한다.
이 데이터셋은 다양한 문서 
---
[결과 3]
This study explores the improvement of work efficiency and expertise by applying Private LLM 
based on Large Language Model (LLM) to the field of clinical trials in medical devices. The Private 
LLM system provides sophisticated and accurate answers based on clinical data and shows its potential 
fo
---
[결과 4]
170   Journal of The Korea Society of Computer and Inform

#### `as_retriever()` 메서드

In [26]:
results = None

In [27]:
results

In [28]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

results = retriever.get_relevant_documents(query)

for idx, doc in enumerate(results, start=1):
    print(f"[결과 {idx}]\n" + doc.page_content[:300])
    print("---")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15456\1585591450.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents(query)


[결과 1]
의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하
기 위해 의료기기 임상시험 전문가로부터 총 158개의 문
서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음
과 같이 분류된다:
 규제 문서 (30%): FDA, EMA, PMDA 가이드라인, 
GCP 문서 등
 교육 자료 (20%): 임상시험 수행자 교육 매뉴얼, 온라
인 강의 자료 등
 프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR 
(Clinical Study Report) 템플릿 등
 의료기기 특화 문서 (15%): 의료기기 임상시험 계획
서,
---
[결과 2]
174   Journal of The Korea Society of Computer and Information 
문서(10%)로 구성되며, 각 분류는 도메인 전문가의 검토를 
통해 정확성과 신뢰성을 확보하였다. 특히, 주요 규제 기
관(FDA, EMA, PMDA) 가이드라인, GCP 문서, 환자 동의
서 템플릿 등은 모델이 국제 표준에 기반하여 학습할 수 
있도록 하였고, 교육 자료와 프로토콜은 실질적인 임상시
험 수행과 데이터 관리 작업에서 발생할 수 있는 질문들에 
대응할 수 있는 기초를 제공한다.
이 데이터셋은 다양한 문서 
---
[결과 3]
This study explores the improvement of work efficiency and expertise by applying Private LLM 
based on Large Language Model (LLM) to the field of clinical trials in medical devices. The Private 
LLM system provides sophisticated and accurate answers based on clinical data and shows its potential 
fo
---
[결과 4]
170   Journal of The Korea Society of Computer and Inform